# First Steps in accessing Satellite Imagery with Sentinel Hub APIs

The Sentinel Hub API is a RESTful API interface that provides access to various satellite imagery archives. It allows you to access raw satellite data, rendered images, statistical analysis, and other features.

In this notebook, we will learn how to:
- Search and discover data using Catalog API
- How to quickly access satellite imagery though Sentinel Hub using Process API.
- Visualising NDVI derived from the satellite imagery

In [ ]:
!pip install sentinelhub matplotlib pandas folium

In [37]:
# Utilities
import os
import matplotlib.pyplot as plt
import pandas as pd
import folium
import numpy as np

# from dotenv import load_dotenv


# EDC libraries
# from edc import setup_environment_variables

from sentinelhub import (
    SHConfig,
    DataCollection,
    SentinelHubCatalog,
    SentinelHubRequest,
    SentinelHubStatistical,
    BBox,
    bbox_to_dimensions,
    CRS,
    MimeType,
    Geometry,
)

In [78]:
import matplotlib.pyplot as plt
import numpy as np


def plot_image(
    image: np.ndarray,
    factor: float = 1.0,
    clip_range: float = None,
    **kwargs: None
) -> None:
    """Utility function for plotting RGB images."""
    fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(15, 15))
    if clip_range is not None:
        ax.imshow(np.clip(image * factor, *clip_range), **kwargs)
    else:
        ax.imshow(image * factor, **kwargs)
    ax.set_xticks([])
    ax.set_yticks([])

# Tool for your polygons visualisation

def geometry_center(geometry):
    coords = []

    if geometry["type"] == "Polygon":
        coords = geometry["coordinates"][0]

    elif geometry["type"] == "MultiPolygon":
        for poly in geometry["coordinates"]:
            coords.extend(poly[0])

    lons, lats = zip(*coords)
    return sum(lats) / len(lats), sum(lons) / len(lons)

def folium_polygon_visualisation(area, data_type='bbox'):
    
    if data_type == 'bbox':
        min_lon, min_lat, max_lon, max_lat = area

        center_lat = (min_lat + max_lat) / 2
        center_lon = (min_lon + max_lon) / 2

        m = folium.Map(location=[center_lat, center_lon], zoom_start=10)

        folium.Rectangle(
            bounds=[[min_lat, min_lon], [max_lat, max_lon]],
            color="red",
            weight=2,
            fill=True,
            fill_opacity=0.2,
        ).add_to(m)
    
    elif data_type == 'polygon':
        center_lat, center_lon = geometry_center(area)
        m = folium.Map(location=[center_lat, center_lon], zoom_start=14)
        folium.GeoJson(
        {
            "type": "Feature",
            "geometry": area,
            "properties": {}
        },
        
    ).add_to(m)
    return m
    


## Credentials


Go to "User settings" > "OAuth clients" and create a new credentials

https://apps.sentinel-hub.com/dashboard/#/account/settings

In [39]:
# load_dotenv()

# Pass Sentinel Hub credentials to SHConfig
config = SHConfig()
config.sh_client_id = 'your_client_id_here'
config.sh_client_secret = 'your_client_secret_here'

## Setting an area of interest

The bounding box in `WGS84` coordinate system is `[(longitude and latitude coordinates of lower left and upper right corners)]`. You can get the bbox for a different area at the [bboxfinder](http://bboxfinder.com/) website.

All requests require a bounding box to be given as an instance of `sentinelhub.geometry.BBox` with corresponding Coordinate Reference System (`sentinelhub.constants.CRS`). In our case it is in WGS84 and we can use the predefined WGS84 coordinate reference system from `sentinelhub.constants.CRS`.

In [40]:
aoi_coords_wgs84 = [15.461282, 46.757161, 15.574922, 46.851514]

When the bounding box bounds have been defined, you can initialize the `BBox` of the area of interest. Using the `bbox_to_dimensions` utility function, you can provide the desired resolution parameter of the image in meters and obtain the output image shape.

In [ ]:
resolution = 10
aoi_bbox = BBox(bbox=aoi_coords_wgs84, crs=CRS.WGS84)
aoi_size = bbox_to_dimensions(aoi_bbox, resolution=resolution)

print(f"Image shape at {resolution} m resolution: {aoi_size} pixels")

In [ ]:
# Plot the bounding box on a map
m = folium_polygon_visualisation(aoi_coords_wgs84)
m

## Catalog API
To search and discover data, you can use the Catalog API. Sentinel Hub Catalog API (or shortly "Catalog") is an API implementing the STAC Specification, providing geospatial information for data available in Sentinel Hub. Firstly, to initialise the `SentinelHubCatalog` class we will use:

In [9]:
catalog = SentinelHubCatalog(config=config)

Now we can build the Catalog API request; to do this we use the `aoi_bbox` we defined earlier as well as `time_interval` and insert these into the request:

In [ ]:
aoi_bbox = BBox(bbox=aoi_coords_wgs84, crs=CRS.WGS84)
time_interval = "2019-05-01", "2019-06-20"

search_iterator = catalog.search(
    DataCollection.SENTINEL2_L2A,
    bbox=aoi_bbox,
    time=time_interval,
    fields={"include": ["id", "properties.datetime"], "exclude": []},
)

results = list(search_iterator)
print("Total number of results:", len(results))

results

## Process API

### Example 1: True Color Image

We build the request according to the [API Reference](https://docs.sentinel-hub.com/api/latest/), using the `SentinelHubRequest` class. Each Process API request also needs an [evalscript](https://docs.sentinel-hub.com/api/latest/evalscript/). An evalscript (or "custom script") is a piece of Javascript code which defines how the satellite data shall be processed by Sentinel Hub and what values the service shall return. It is a required part of any [process](https://docs.sentinel-hub.com/api/latest/api/process/), [batch processing](https://docs.sentinel-hub.com/api/latest/api/batch/) or [OGC request](https://docs.sentinel-hub.com/api/latest/api/ogc/).

The information that we specify in the `SentinelHubRequest` object is:
- an evalscript,
- a list of input data collections with time interval,
- a format of the response,
- a bounding box and its size (size or resolution).
- `mosaickingOrder` (optional): in this example we have used `leastCC` which will return pixels from the least cloudy acquisition in the specified time period.

The evalscript in the example is used to select the appropriate bands. We return the RGB (B04, B03, B02) Sentinel-2 L2A bands.

The least cloudy image from the time period is downloaded. Without any additional parameters in the evalscript, the downloaded data will correspond to reflectance values in `UINT8` format (values in 0-255 range).

In [14]:
evalscript_true_color = """
//VERSION=3

    function setup() {
        return {
            input: [{
                bands: ["B02", "B03", "B04"]
            }],
            output: {
                bands: 3,
            }
        };
    }

    function evaluatePixel(sample) {
        return [sample.B04, sample.B03, sample.B02];
    }
"""

request_true_color = SentinelHubRequest(
    evalscript=evalscript_true_color,
    input_data=[
        SentinelHubRequest.input_data(
            data_collection=DataCollection.SENTINEL2_L2A,
            time_interval=("2022-06-01", "2022-08-20")        )
    ],
    responses=[SentinelHubRequest.output_response("default", MimeType.TIFF)],
    bbox=aoi_bbox,
    size=aoi_size,
    config=config,
)

The method `get_data()` will always return a list of length 1 with the available image from the requested time interval in the form of numpy arrays.

In [ ]:
true_color_imgs = request_true_color.get_data()

In [ ]:
print(
    f"Returned data is of type = {type(true_color_imgs)} and length {len(true_color_imgs)}."
)
print(
    f"Single element in the list is of type {type(true_color_imgs[-1])} and has shape {true_color_imgs[-1].shape}"
)

In [ ]:
image = true_color_imgs[0]
print(f"Image type: {image.dtype}")

# plot function
# factor 1/255 to scale between 0-1
# factor 3.5 to increase brightness
plot_image(image, factor=3.5 / 255, clip_range=(0, 1))

In [ ]:
image
# Band non RGB

### Example 2: NDVI Image

Secondly, we will also show you an example of how to calculate and visualise NDVI using the same API. NDVI is a very commonly used spectral vegetation index for vegetation monitoring, for example, monitoring crop growth and yields. As you will notice in the codeblock below, the evalscript has changed substantially:
- we are only using Band 4 and Band 8 as an input into our script.
- In the `evaluatePixel()` function, we calculate NDVI and visualise this using the `imgVals` array. 

In [19]:
evalscript_ndvi = """
//VERSION=3
function setup() {
  return {
    input: [{
      bands: [
        "B04",
        "B08",
        "dataMask"
      ]
    }],
    output: {
      bands: 4
    }
  }
}


function evaluatePixel(sample) {
    let val = (sample.B08 - sample.B04) / (sample.B08 + sample.B04);
    let imgVals = null;

    if (val<-1.1) imgVals = [0,0,0];
    else if (val<-0.2) imgVals = [0.75,0.75,0.75];
    else if (val<-0.1) imgVals = [0.86,0.86,0.86];
    else if (val<0) imgVals = [1,1,0.88];
    else if (val<0.025) imgVals = [1,0.98,0.8];
    else if (val<0.05) imgVals = [0.93,0.91,0.71];
    else if (val<0.075) imgVals = [0.87,0.85,0.61];
    else if (val<0.1) imgVals = [0.8,0.78,0.51];
    else if (val<0.125) imgVals = [0.74,0.72,0.42];
    else if (val<0.15) imgVals = [0.69,0.76,0.38];
    else if (val<0.175) imgVals = [0.64,0.8,0.35];
    else if (val<0.2) imgVals = [0.57,0.75,0.32];
    else if (val<0.25) imgVals = [0.5,0.7,0.28];
    else if (val<0.3) imgVals = [0.44,0.64,0.25];
    else if (val<0.35) imgVals = [0.38,0.59,0.21];
    else if (val<0.4) imgVals = [0.31,0.54,0.18];
    else if (val<0.45) imgVals = [0.25,0.49,0.14];
    else if (val<0.5) imgVals = [0.19,0.43,0.11];
    else if (val<0.55) imgVals = [0.13,0.38,0.07];
    else if (val<0.6) imgVals = [0.06,0.33,0.04];
    else imgVals = [0,0.27,0];


    imgVals.push(sample.dataMask)

    return imgVals
}
"""

request_ndvi_img = SentinelHubRequest(
    evalscript=evalscript_ndvi,
    input_data=[
        SentinelHubRequest.input_data(
            data_collection=DataCollection.SENTINEL2_L2A,
            time_interval=("2022-05-01", "2022-05-20"),
            other_args={"dataFilter": {"mosaickingOrder": "leastCC"}},
        )
    ],
    responses=[SentinelHubRequest.output_response("default", MimeType.PNG)],
    bbox=aoi_bbox,
    size=aoi_size,
    config=config,
)

The same method as before is used to request and then visualise the data. In the visualisation, the darker greens indicate a higher NDVI value (vegetation, forest) and the lighter greens (urban areas and water bodies) represent areas with lower NDVI values.

In [20]:
ndvi_img = request_ndvi_img.get_data()

In [ ]:
print(
    f"Returned data is of type = {type(ndvi_img)} and length {len(ndvi_img)}."
)
print(
    f"Single element in the list is of type {type(ndvi_img[-1])} and has shape {ndvi_img[-1].shape}"
)

In [ ]:
image = ndvi_img[0]
print(f"Image type: {image.dtype}")

# plot function
plot_image(image, factor=1 / 255)

In [ ]:
ndvi_img[0]

## Example 3. Raw image

In [24]:
evalscript_raw = """
//VERSION=3
function setup() {
    return {
        input: [{
            bands: ["B01","B02","B03","B04","B05","B06","B07","B08","B8A","B09", "B11","B12","SCL","CLM"],
            units: "DN"
        }],
        output: {
            bands: 14,
            sampleType: "UINT16"
        }
    };
}

function updateOutputMetadata(scenes, inputMetadata, outputMetadata) {
    outputMetadata.userData = { "norm_factor":  inputMetadata.normalizationFactor,
                                "scenes":  JSON.stringify(scenes)}
}
function evaluatePixel(sample) {
  if (sample.CLM == 1) {
    return [sample.B01, sample.B02, sample.B03, sample.B04, sample.B05, sample.B06,
            sample.B07, sample.B08, sample.B8A, sample.B09, sample.B11, sample.B12, sample.SCL, 1]
  }
  return [sample.B01, sample.B02, sample.B03, sample.B04, sample.B05, sample.B06,
            sample.B07, sample.B08, sample.B8A, sample.B09, sample.B11, sample.B12, sample.SCL, 0];
}
"""

request_raw_img = SentinelHubRequest(
    evalscript=evalscript_raw,
    input_data=[
        SentinelHubRequest.input_data(
            data_collection=DataCollection.SENTINEL2_L2A,
            time_interval=("2022-05-01", "2022-05-20"),
            other_args={"dataFilter": {"mosaickingOrder": "leastCC"}},
        )
    ],
    responses=[SentinelHubRequest.output_response("default", MimeType.TIFF)],
    bbox=aoi_bbox,
    size=aoi_size,
    config=config,
)

In [25]:
raw_img = request_raw_img.get_data()

In [ ]:
raw_img

In [ ]:
image = raw_img[0]
print(f"Image type: {image.dtype}")

# plot function
plot_image(image[:, :, 0:3], factor=1 / 1000)

## Statistical API

In the Process API examples, we have seen how to obtain satellite imagery. [Statistical API](https://docs.sentinel-hub.com/api/latest/api/statistical/) can be used in a very similar way. The main difference is that the results of Statistical API are aggregated statistical values of satellite data instead of entire images. In many use cases, such values are all that we need. By using Statistical API we can avoid downloading and processing large amounts of satellite data.

All general rules for building evalscripts apply. However, there are some specifics when using evalscripts with the Statistical API:

- The `evaluatePixel()` function must, in addition to other output, always return a `dataMask` output. This output defines which pixels are excluded from calculations. For more details and an example, see [here](https://docs.sentinel-hub.com/api/latest/api/statistical/#exclude-pixels-from-calculations-datamask-output).
- The default value of sampleType is `FLOAT32`.
- The output.bands parameter in the setup() function can be an array. This makes it possible to specify custom names for the output bands and different output `dataMask` for different outputs, see this [example](https://docs.sentinel-hub.com/api/latest/api/statistical/examples/#multiple-outputs-with-different-datamasks-multi-band-output-with-custom-bands-names-and-different-histogram-types).

### Requesting, and plotting an NDVI time series for a single field

In the example here, we will calculate NDVI for a specific field of interest and then plot the mean NDVI and standard deviation over the requested time period. First we define our evalscript:

In [50]:
evalscript = """
//VERSION=3
function setup() {
  return {
    input: [{
      bands: [
        "B04",
        "B08",
        "dataMask"
      ]
    }],
    output: [
      {
        id: "ndvi",
        bands: 1
      },
      {
        id: "dataMask",
        bands: 1
      }]
  };
}

function evaluatePixel(samples) {
    let index = (samples.B08 - samples.B04) / (samples.B08 + samples.B04);
    return {
        ndvi: [index],
        dataMask: [samples.dataMask],
    };
}

"""

In this example, we will compare two fields within the area we requested using Process API:

In [51]:
field1 = {"type":"Polygon","coordinates":[[[15.541723001099184,46.820368115848446],
                                           [15.541756949727985,46.82037740810231],
                                           [15.54192669287196,46.82008470133467],
                                           [15.542211861353849,46.81964331510048],
                                           [15.539394125163792,46.81905789197882],
                                           [15.539251540922846,46.819805931503055],
                                           [15.541723001099184,46.820368115848446]]]}

field2 = {"type":"Polygon","coordinates":[[[15.507170086710744,46.83938135202761],
                                           [15.508086699688228,46.83921879483953],
                                           [15.50831755036404,46.839576420004114],
                                           [15.508582349668648,46.83992939835186],
                                           [15.508874307876296,46.840221997066486],
                                           [15.50860950857169,46.840514594187695],
                                           [15.50842618597619,46.84082112279607],
                                           [15.508113858591262,46.840639992466144],
                                           [15.50781511065786,46.84039384001332],
                                           [15.50739414766079,46.83981328730921],
                                           [15.507149717533464,46.83939064099493],
                                           [15.507170086710744,46.83938135202761]]]}

In [52]:
fields = {
  "type": "MultiPolygon",
  "coordinates": [
    [
      [
        [
          15.541723001099184,
          46.820368115848446
        ],
        [
          15.541756949727985,
          46.82037740810231
        ],
        [
          15.54192669287196,
          46.82008470133467
        ],
        [
          15.542211861353849,
          46.81964331510048
        ],
        [
          15.539394125163792,
          46.81905789197882
        ],
        [
          15.539251540922846,
          46.819805931503055
        ],
        [
          15.541723001099184,
          46.820368115848446
        ]
      ]
    ],
    [
      [
        [
          15.507170086710744,
          46.83938135202761
        ],
        [
          15.508086699688228,
          46.83921879483953
        ],
        [
          15.50831755036404,
          46.839576420004114
        ],
        [
          15.508582349668648,
          46.83992939835186
        ],
        [
          15.508874307876296,
          46.840221997066486
        ],
        [
          15.50860950857169,
          46.840514594187695
        ],
        [
          15.50842618597619,
          46.84082112279607
        ],
        [
          15.508113858591262,
          46.840639992466144
        ],
        [
          15.50781511065786,
          46.84039384001332
        ],
        [
          15.50739414766079,
          46.83981328730921
        ],
        [
          15.507149717533464,
          46.83939064099493
        ],
        [
          15.507170086710744,
          46.83938135202761
        ]
      ]
    ]
  ]
}

In [ ]:
folium_polygon_visualisation(fields, data_type='polygon')

In [ ]:
!pip install --upgrade jupyterlab ipython


Now we have defined the evalscript and the two fields of interest, we can build the first Statistical API Request, before returning the response for the first field. In this request, as part of the payload we define some input parameters:
- `time_interval` this defines the time range of our request.
- `aggregation_interval` this defines the length of time each interval is. In this case, the interval is 10 days. he aggregation intervals should be at least one day long (e.g. "P5D", "P30D"). You can only use period OR time designator not both. 
- `dataFilter: {maxCloudCoverage}` this is an additional argument in our request which filters out image acquisitions that have a cloud coverage percentage above 10%.

**NOTE:**
If a timeRange is not divisible by an aggregationInterval, the last ("not full") time interval will be dismissed by default (SKIP option). The user can instead set the lastIntervalBehavior to SHORTEN (shortens the last interval so that it ends at the end of the provided time range) or EXTEND (extends the last interval over the end of the provided time range so that all the intervals are of equal duration).

In [ ]:
geometry = Geometry(geometry=fields, crs=CRS.WGS84)

request = SentinelHubStatistical(
    aggregation=SentinelHubStatistical.aggregation(
        evalscript=evalscript,
        time_interval=('2022-04-01T00:00:00Z', '2022-08-30T23:59:59Z'),
        aggregation_interval='P10D',
        size=[368.043, 834.345],
    ),
    input_data=[
        SentinelHubStatistical.input_data(
            DataCollection.SENTINEL2_L1C,
            other_args={"dataFilter": {"maxCloudCoverage": 10}},
      ),
    ],
    geometry=geometry,
    config=config
)

response1 = request.get_data()
response1

However, as it is clear to see, our response is not that useful in `json` format. It's difficult to read from a human perspective. So, let's transform it into a `pandas` dataframe. To help us achieve this, let's define some helper functions. 

In [54]:
# define functions to extract statistics for all acquisition dates
def extract_stats(date, stat_data):
    d = {}
    for key, value in stat_data['outputs'].items():
        stats = value['bands']['B0']['stats']
        if stats['sampleCount']==stats['noDataCount']:
            continue
        else:
            d['date'] = [date]
            for stat_name, stat_value in stats.items():
                if (stat_name=='sampleCount' or stat_name=='noDataCount'):
                    continue
                else:
                    d[f'{key}_{stat_name}'] = [stat_value]
    return pd.DataFrame(d)

def read_acquisitions_stats(stat_data):
    df_li = []
    for aq in stat_data:
        date = aq['interval']['from'][:10]
        df_li.append(extract_stats(date, aq))
    return pd.concat(df_li)

In [ ]:
result_df1 = read_acquisitions_stats(response1[0]['data'])
result_df1

We can take this another step further, and display the data in a time series using the Matplotlib python library:

In [ ]:
fig_stat, ax_stat =  plt.subplots(1, 1, figsize=(12,6))
t1 = result_df1['date']
ndvi_mean_field1 = result_df1['ndvi_mean']
ndvi_std_field1 = result_df1['ndvi_stDev']
ax_stat.plot(t1, ndvi_mean_field1, label='field 1 mean')
ax_stat.fill_between(t1, ndvi_mean_field1 - ndvi_std_field1, ndvi_mean_field1 + ndvi_std_field1, alpha=0.3, label='field 1 stDev')
ax_stat.tick_params(axis='x', labelrotation=30, labelsize=12)
ax_stat.tick_params(axis='y', labelsize=12)
ax_stat.set_xlabel('Date', size=15)
ax_stat.set_ylabel('NDVI/unitless', size=15)
ax_stat.legend(loc='lower right', prop={'size': 12})
ax_stat.set_title('NDVI time series', fontsize=20)
for label in ax_stat.get_xticklabels()[1::2]:
    label.set_visible(False)

### Comparing different fields

Now that we have learnt how to plot the data for the first field, let's take this another step forward and compare the NDVI time series of the first field with the second field. We will now run the same request for our second field and then transform the response into a second Pandas dataframe.

In [57]:
geometry = Geometry(geometry=field2, crs=CRS.WGS84)

request = SentinelHubStatistical(
    aggregation=SentinelHubStatistical.aggregation(
        evalscript=evalscript,
        time_interval=('2022-04-01T00:00:00Z', '2022-08-30T23:59:59Z'),
        aggregation_interval='P10D',
        size=[368.043, 834.345],
    ),
    input_data=[
        SentinelHubStatistical.input_data(
            DataCollection.SENTINEL2_L1C,
            other_args={"dataFilter": {"maxCloudCoverage": 10}},
      ),
    ],
    geometry=geometry,
    config=config
)

response2 = request.get_data()
result_df2 = read_acquisitions_stats(response2[0]['data'])

Now we have requested the statistics for both fields and transformed them into Pandas dataframes, let's plot the two time series and visualise this in the same plot:

In [ ]:
fig_stat, ax_stat =  plt.subplots(1, 1, figsize=(12,6))
t1 = result_df1['date']
t2 = result_df1['date']
ndvi_mean_field1 = result_df1['ndvi_mean']
ndvi_std_field1 = result_df1['ndvi_stDev']
ndvi_mean_field2 = result_df2['ndvi_mean']
ndvi_std_field2 = result_df2['ndvi_stDev']
ax_stat.plot(t1, ndvi_mean_field1, label='field 1 mean')
ax_stat.fill_between(t1, ndvi_mean_field1 - ndvi_std_field1, ndvi_mean_field1 + ndvi_std_field1, alpha=0.3, label='field 1 stDev')
ax_stat.plot(t2, ndvi_mean_field2, label='field 2 mean')
ax_stat.fill_between(t2, ndvi_mean_field2 - ndvi_std_field2, ndvi_mean_field2 + ndvi_std_field2, alpha=0.3, label='field 2 stDev')
ax_stat.tick_params(axis='x', labelrotation=30, labelsize=12)
ax_stat.tick_params(axis='y', labelsize=12)
ax_stat.set_xlabel('Date', size=15)
ax_stat.set_ylabel('NDVI/unitless', size=15)
ax_stat.legend(loc='lower right', prop={'size': 12})
ax_stat.set_title('NDVI time series', fontsize=20)
for label in ax_stat.get_xticklabels()[1::2]:
    label.set_visible(False)

## Summary

So what have we learnt in this notebook?

- Search and discover data using Catalog API
- How to quickly access satellite imagery though Sentinel Hub using Process API.
- Using Statistical API to produce NDVI time series for single and multiple fields.

This concludes this notebook on working with Sentinel Hub APIs to access satellite imagery. For more information you can check out the [Sentinel Hub API](https://docs.sentinel-hub.com/api/latest/) Documentation and the [Sentinel Hub Python package](https://sentinelhub-py.readthedocs.io/en/latest/index.html) documentation too.